# Blue-chip Legacy Upgrade Engine: live demo
*Keeping your models out of the red.* | EXL x Trinity Data & AI Hackathon 2026 | Challenge 04 | Team **Blue**

A bank has hundreds of legacy models. Validating one takes weeks, its documentation is often out of date, and rewriting it
(here: R to Python) is risky. **Blue automates the evidence and a human makes the decision.**

**How to use this notebook (for the video):** run the cells top to bottom.
- The **setup cell** loads the repo (on Colab you upload `blue-hackathon.zip` once).
- The **pipeline cell** (section 1) runs everything and takes under a minute. Start it, then talk over it.
- Everything after that is instant and shows one stage at a time.
- The last section deliberately **tries to break the guardrails**, which is the best bit to show on camera.

> **Honest note:** this runs in `offline` mode. No LLM is called. The documentation, the R to Python drafts and the
> report text come from labelled rule-based stand-ins. What is real is the machinery around them: the evidence checker,
> the parity test against genuine R output, the validation maths and the guardrails. That machinery is what makes an
> LLM safe to plug in.

## 0. Setup

In [ ]:
# 1. Get the Blue repo into this notebook and make sure its libraries are installed (about 20 s on Colab).
import importlib.util, os, subprocess, sys, warnings, zipfile
from pathlib import Path

warnings.filterwarnings("ignore", category=DeprecationWarning)
IN_COLAB = "google.colab" in sys.modules
GITHUB_URL = ""  # optional: paste your repo URL here instead of uploading the zip


def find_repo():
    here = Path.cwd()
    candidates = [here, here.parent, *Path("/content").glob("*"), *Path("/content").glob("*/*")]
    for p in candidates:
        if (p / "config.yaml").exists() and (p / "blue" / "pipeline.py").exists():
            return p


REPO = find_repo()
if REPO is None and GITHUB_URL:
    subprocess.check_call(["git", "clone", "-q", GITHUB_URL, "/content/blue"])
    REPO = find_repo()
if REPO is None and IN_COLAB:
    from google.colab import files

    print("Choose blue-hackathon.zip ...")
    uploaded = next(iter(files.upload()))
    with zipfile.ZipFile(uploaded) as z:
        z.extractall("/content")
    REPO = find_repo()
if REPO is None:
    raise RuntimeError(
        "Could not find the Blue repo. On Colab, upload blue-hackathon.zip when asked; "
        "locally, open this notebook from the repo's notebooks/ folder."
    )
os.chdir(REPO)
sys.path.insert(0, str(REPO))

NEEDED = {
    "fairlearn": "fairlearn",
    "statsmodels": "statsmodels",
    "yaml": "pyyaml",
    "jinja2": "jinja2",
    "pydantic": "pydantic",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
    "matplotlib": "matplotlib",
}
missing = [pkg for mod, pkg in NEEDED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Repo:", REPO)
print("Installed just now:", missing or "nothing, everything was already available")

## 1. Run the whole pipeline
Six stages run in sequence on three legacy models (two R, one Python), using public credit-card default data:
**Discover, Document, Redevelop, Validate, Report, Monitor**. The prints below show each stage as it finishes.
In offline mode there is no network call and no API key.

In [ ]:
import copy, html, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, Markdown, display

from blue.config import load_config, resolve
from blue.pipeline import run

# Blue's report module switches matplotlib to a headless backend (it only needs PNG bytes); this turns notebook charts back on.
get_ipython().run_line_magic("matplotlib", "inline")

cfg = load_config()
ART = resolve(cfg, "artifacts_dir") / "latest"
RERUN = True  # on retakes set this to False to reuse the last run's files instead of waiting about a minute

if RERUN or not (ART / "metrics.json").exists():
    out = run(cfg, mode="offline", use_mlflow=False)
else:
    print("Re-using the previous run in", ART)


def load(name):
    return json.loads((ART / name).read_text(encoding="utf-8"))

## 2. Discover: what models do we have, and how risky are they?
Blue scans the code and extracts facts: language, algorithm, number of predictors, decision cut-off. It then assigns a
**risk tier with a rule you can read**: high materiality means Tier 1, and medium materiality or a complex algorithm means Tier 2.
Materiality is a *business input* from the config file, because it cannot be inferred from code. (SS1/23 Principle 1.)

In [ ]:
inv = pd.DataFrame(load("inventory.json"))
display(
    inv[
        [
            "model_id",
            "language",
            "algorithm",
            "n_predictors",
            "materiality",
            "complexity_score",
            "risk_tier",
        ]
    ].style.hide(axis="index")
)
print("Why credit_pd is Tier 1:\n ", inv.loc[inv.model_id == "credit_pd", "tier_rationale"].iloc[0])

## 3. Document: claims you can check, not text you must trust
Every sentence in the generated documentation **quotes the code lines it relies on**, and a verifier confirms the quote
really exists in the file. Then the **stale-documentation detector** compares the *old* document with what the code actually does.
(SS1/23 Principle 3.)

In [ ]:
lines = (ART / "docs" / "credit_pd.md").read_text(encoding="utf-8").splitlines()
end = next(i for i, line in enumerate(lines) if line.startswith("## Variables"))
display(Markdown("\n".join(lines[:end])))

In [ ]:
disc = pd.DataFrame(load("docs/credit_pd_discrepancies.json"))
disc["code evidence"] = disc["evidence"].apply(
    lambda ev: ", ".join(
        f"{Path(e['file']).name}:L{e['start_line']}"
        + (f"-{e['end_line']}" if e["end_line"] != e["start_line"] else "")
        for e in ev
    )
)
print("The OLD documentation of credit_pd versus what the code really does:")
display(
    disc[["field", "documented", "actual", "severity", "code evidence"]].style.hide(axis="index")
)
control = load("docs/pricing_glm_discrepancies.json")
print(
    f"Control: pricing_glm has correct documentation, and Blue reports {len(control)} discrepancies (no false alarms)."
)

## 4. Redevelop: port R to Python and *prove* it matches
The R model was really run in R (version 4.6.2) on the same data, and its output is the reference. Blue's Python port must
match it **coefficient by coefficient and prediction by prediction**. When it does not, Blue **diagnoses why** and repairs
the code, so no one has to trust that the port is right. (SS1/23 Principle 3.)

In offline mode the three drafts are scripted; the diagnosis between them is real.

In [ ]:
par = load("parity.json")
rows = [
    {
        "draft": it["n"],
        "written by": it["source"],
        "result": "PASS" if it["passed"] else "FAIL",
        "what the parity test diagnosed": ", ".join(it["diagnosis"]) or "-",
        "checks passed": f'{sum(c["passed"] for c in it["checks"])}/{len(it["checks"])}',
    }
    for it in par["iterations"]
]
display(pd.DataFrame(rows).style.hide(axis="index"))
print(
    "Draft 1 forgot the categorical terms. Draft 2 used a default L2 penalty that shrinks every coefficient."
)
print("What changed from draft 2 to draft 3 (the repair):\n")
print("\n".join((ART / "port" / "draft_2_to_3.diff").read_text(encoding="utf-8").splitlines()[:14]))

In [ ]:
from blue.data import prepare_splits
from blue.fixtures import load_reference
from blue.redevelop import run_candidate, compare

splits = prepare_splits(cfg)
ref = load_reference(
    cfg, "credit_pd", splits
)  # genuine R output; refuses to load if inputs changed
port_file = ART / "port" / "credit_pd_port.py"
cand = run_candidate(
    port_file,
    splits["train"],
    {"test": splits["test"], "edge": splits["edge"]},
    cfg["parity"]["timeout_seconds"],
)

side = pd.DataFrame({"R (reference)": ref["coefficients"], "Python port": cand["coefficients"]})
side["abs difference"] = (side["R (reference)"] - side["Python port"]).abs()
display(
    side.style.format(
        {"R (reference)": "{:.6f}", "Python port": "{:.6f}", "abs difference": "{:.1e}"}
    )
)
print(
    f"Largest coefficient difference: {side['abs difference'].max():.1e}   (tolerance {cfg['parity']['tolerances']['coefficient_max_abs_diff']:.0e})"
)

## 5. Validate: an independent second look
Discrimination, calibration, stability, fairness on protected attributes and a **challenger model**. Sex and age are *not*
inputs to the model, but they are tested for disparate outcomes. The results below were **not tuned to look good**, and two are Red.
(SS1/23 Principle 4.) The status colours always come with a text label.

In [ ]:
m = load("metrics.json")
findings = pd.DataFrame(m["findings"])[["area", "metric", "value", "rag", "criterion"]]
STATUS = {
    "Green": ("#0ca30c", "#ffffff"),
    "Amber": ("#fab219", "#0b0b0b"),
    "Red": ("#d03b3b", "#ffffff"),
}
paint = lambda v: (
    f"background-color:{STATUS[v][0]};color:{STATUS[v][1]};font-weight:600;text-align:center"
    if v in STATUS
    else ""
)
display(findings.style.hide(axis="index").map(paint, subset=["rag"]))
print("Draft outcome:", m["outcome"])

In [ ]:
INK, MUTED, GRID, SURFACE, BLUE, ORANGE = (
    "#0b0b0b",
    "#52514e",
    "#e6e5e1",
    "#fcfcfb",
    "#2a78d6",
    "#eb6834",
)
plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": GRID,
        "axes.labelcolor": MUTED,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "text.color": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "axes.axisbelow": True,
        "font.size": 11,
    }
)

cal = pd.DataFrame(m["performance"]["calibration_table"])
cal["decile"] = cal["bin"] + 1
worst = (cal["mean_pd"] - cal["observed"]).abs().idxmax()

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 4.4), gridspec_kw={"width_ratios": [1.25, 1]})
a1.plot(
    cal["decile"],
    cal["mean_pd"],
    color=BLUE,
    lw=2,
    marker="o",
    ms=8,
    mec=SURFACE,
    mew=1.5,
    label="Predicted PD",
)
a1.plot(
    cal["decile"],
    cal["observed"],
    color=ORANGE,
    lw=2,
    marker="o",
    ms=8,
    mec=SURFACE,
    mew=1.5,
    label="Observed default rate",
)
w = cal.loc[worst]
a1.vlines(
    w["decile"],
    min(w["mean_pd"], w["observed"]),
    max(w["mean_pd"], w["observed"]),
    color=INK,
    lw=1,
    ls=":",
)
a1.annotate(
    f"Worst decile: gap {m['performance']['worst_decile_gap']:.3f}\n(predicted {w['mean_pd']:.3f}, observed {w['observed']:.3f})",
    xy=(w["decile"], (w["mean_pd"] + w["observed"]) / 2),
    xytext=(4.2, 0.50),
    textcoords="data",
    ha="left",
    va="center",
    color=INK,
    fontsize=10,
    arrowprops=dict(arrowstyle="-", color=MUTED, lw=1),
)
a1.set_title("Calibration by risk decile", loc="left", fontweight="bold")
a1.set_xlabel("Risk decile (10 = highest predicted PD)")
a1.set_ylabel("Default probability / rate")
a1.set_xticks(range(1, 11))
a1.legend(frameon=False, loc="upper left")

order = ["<30", "30-44", "45+"]
age = {g["group"]: g for g in m["fairness"]["AGE"]["groups"]}
vals = [age[g]["tpr_declined_defaulters"] for g in order]
bars = a2.bar(order, vals, width=0.38, color=BLUE)
for b, v in zip(bars, vals):
    a2.text(
        b.get_x() + b.get_width() / 2, v + 0.01, f"{v:.3f}", ha="center", color=INK, fontsize=10
    )
a2.set_ylim(0, max(vals) * 1.25)
a2.set_title("Defaulters declined, by age band", loc="left", fontweight="bold")
a2.set_xlabel(
    f"Equalised-odds gap (AGE): {m['fairness']['AGE']['equalized_odds_gap']:.3f}  (Amber)"
)
a2.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

## 6. Report: one document for the validation committee
Every number in the narrative must match a computed metric, otherwise the text is rejected (you will see that happen in
section 8). Approvals in this demo are recorded as `auto-demo`, never as a human, and the report says so.

In [ ]:
report_html = (ART / "report.html").read_text(encoding="utf-8")
with warnings.catch_warnings():
    warnings.simplefilter(
        "ignore"
    )  # IPython suggests IFrame(); that only takes a URL, we embed the file itself
    display(
        HTML(
            f'<iframe srcdoc="{html.escape(report_html, quote=True)}" width="100%" height="760" '
            'style="border:1px solid #ccc;border-radius:6px"></iframe>'
        )
    )
print("Full file:", ART / "report.html")

## 7. Monitor: when does the model need re-validating?
A **synthetic** stress scenario: borrower behaviour worsens batch by batch. Blue tracks drift (PSI for the score, CSI for the
inputs) and the decline rate, and **raises a re-validation ticket** once thresholds are crossed. This shows the mechanism,
not real model decay, because the data has no time dimension.

In [ ]:
mon = load("monitor.json")
b = pd.DataFrame(mon["batches"])
MONSTATUS = {
    "OK": ("#0ca30c", "#ffffff"),
    "Watch": ("#fab219", "#0b0b0b"),
    "Re-validate": ("#d03b3b", "#ffffff"),
}
paint2 = lambda v: (
    f"background-color:{MONSTATUS[v][0]};color:{MONSTATUS[v][1]};font-weight:600;text-align:center"
    if v in MONSTATUS
    else ""
)
display(
    b[["batch", "score_psi", "max_csi", "max_csi_feature", "decline_rate", "status"]]
    .style.hide(axis="index")
    .map(paint2, subset=["status"])
    .format({"score_psi": "{:.3f}", "max_csi": "{:.3f}", "decline_rate": "{:.3f}"})
)

th = mon["thresholds"]
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(
    b["batch"],
    b["score_psi"],
    color=BLUE,
    lw=2,
    marker="o",
    ms=8,
    mec=SURFACE,
    mew=1.5,
    label="Score PSI",
)
ax.plot(
    b["batch"],
    b["max_csi"],
    color=ORANGE,
    lw=2,
    marker="o",
    ms=8,
    mec=SURFACE,
    mew=1.5,
    label="Largest input CSI",
)
for key, name in (("psi_watch", "watch"), ("psi_revalidate", "re-validate")):
    ax.axhline(th[key], color=MUTED, lw=1, ls="--")
    ax.text(
        0.85,
        th[key] + 0.012,
        f"{name} threshold {th[key]:.2f}",
        va="bottom",
        ha="left",
        color=MUTED,
        fontsize=10,
    )
ax.set_xlim(0.8, b["batch"].max() + 0.3)
ax.set_xticks(b["batch"])
ax.set_xlabel("Monitoring batch")
ax.set_ylabel("Index value")
ax.set_title(
    "Drift per batch against the thresholds (synthetic scenario)", loc="left", fontweight="bold"
)
ax.legend(frameon=False, loc="upper left")
plt.tight_layout()
plt.show()

print(
    mon["forecast"]
)  # the descriptive trend sentence (the key name is historical; it is not a forecast)
print("\nRe-validation ticket raised automatically:")
print(json.dumps(load("revalidation_ticket.json"), indent=2))

## 8. Try to break it
An LLM, or a tired human, will sometimes produce something wrong. Each cell below **plants a mistake** and shows the guardrail
catching it. These are the same checks the test suite runs.

### 8a. A documentation claim whose evidence is not in the code

In [ ]:
from blue import docgen
from blue.inventory import scan

facts = next(f for f in scan(cfg) if f.model_id == "credit_pd")
source = {facts.path: (REPO / facts.path).read_text(encoding="utf-8")}

honest = docgen.verify_claims(docgen.offline_draft(facts), source)
print(
    f"Honest draft : {sum(c.verified for c in honest.claims)}/{len(honest.claims)} claims verified"
)

forged = docgen.offline_draft(facts)
forged.claims[1].evidence[
    0
].quote = "the model is gradient boosting with 200 trees"  # not in the code
forged = docgen.verify_claims(forged, source)
bad = forged.claims[1]
print(f"Claim         : {bad.text}")
print(f'Evidence given: "{bad.evidence[0].quote}"')
print(f"Verified      : {bad.verified}   <- that quote does not appear in credit_pd.R")
print("The report prints this claim as [UNVERIFIED] instead of presenting it as fact.")

### 8b. A report sentence containing a number nobody computed

In [ ]:
from blue.report import numeric_leaves, ungrounded_numbers

allowed = numeric_leaves(m)
gini = m["performance"]["gini"]
honest_sentence = f"On the test sample the Gini coefficient is {gini:.3f}."
invented_sentence = "On the test sample the Gini coefficient is 0.612."
print(
    "Honest  :",
    honest_sentence,
    "->",
    ungrounded_numbers(honest_sentence, allowed) or "every number traced to a computed metric",
)
print(
    "Invented:",
    invented_sentence,
    "->",
    ungrounded_numbers(invented_sentence, allowed),
    "not found in any metric, so the narrative is rejected",
)

### 8c. Generated code that tries to do something it should not

In [ ]:
from blue.redevelop import check_code

attempts = {
    "ordinary modelling code": "import numpy as np\nimport statsmodels.api as sm\n",
    "imports the os module": "import os\nos.system('echo hi')\n",
    "reads a file it was not given": "import pandas as pd\npd.read_csv('secret.csv')\n",
}
for name, snippet in attempts.items():
    problems = check_code(snippet)
    print(f"{'BLOCKED' if problems else 'allowed':8} {name:32} {problems[:1] if problems else ''}")

### 8d. A port that is *almost* right
Every prediction of the real, passing port is nudged by 0.001. A human reviewing the numbers would never notice;
the parity test does.

In [ ]:
y_test = pd.read_csv(splits["test"])["default"].to_numpy()
verdict = lambda checks: "PASS" if all(c.passed for c in checks) else "FAIL"

checks, diag = compare(ref, cand, y_test, facts.cutoff, cfg)
print("The real port                  :", verdict(checks), diag or "")

nudged = copy.deepcopy(cand)
nudged["predictions"]["test"] = [
    None if v is None else v + 1e-3 for v in cand["predictions"]["test"]
]
checks, diag = compare(ref, nudged, y_test, facts.cutoff, cfg)
print("Same port, predictions +0.001  :", verdict(checks), diag)

## What you just saw, and what it does not claim
- **Real:** the inventory and tiering rule, the evidence checker, the stale-document findings, the parity test against genuine
  R output, the validation results (including the Red ones), the drift monitor and every guardrail in section 8.
- **Scripted stand-ins (offline mode):** the documentation wording, the three port drafts and the report prose. With an API key,
  `--mode live` lets an LLM write these, and the same guardrails then check its work. The live path is written but has
  not yet been run against a real provider.
- **Limits:** one public dataset, one model redeveloped end to end, illustrative thresholds, synthetic drift, fairness tested one attribute at a time.
- **Human in the loop:** every output is a draft until a named validator signs it off.

Repo, tests and the one-page Responsible AI note are in the zip. Start with `README.md`.